# 03 - Smoke-100 Training (Colab + Kaggle)
100 files, leakage-safe Pipeline, XGBoost + Optuna (5 trials), MLflow, SHAP check.

In [ ]:
import os, sys
ON_KAGGLE = os.path.isdir('/kaggle/working')
ON_COLAB = os.path.isdir('/content/drive')
ROOT = '/kaggle/working/repo' if ON_KAGGLE else ('/content/drive/MyDrive/DeepFakeVoiceResearch' if ON_COLAB else os.getcwd())
print(f'Kaggle={ON_KAGGLE} Colab={ON_COLAB} root={ROOT}')
if ROOT not in sys.path: sys.path.insert(0, ROOT)
print('run: pip install -q -r requirements.txt  # first time only')


In [ ]:
import yaml
with open(os.path.join(ROOT,'configs/features.yaml')) as f: feat_cfg=yaml.safe_load(f)
with open(os.path.join(ROOT,'configs/models.yaml')) as f: model_cfg=yaml.safe_load(f)
WORK = '/kaggle/working' if ON_KAGGLE else ROOT
OUT_MODELS = os.path.join(WORK,'models'); OUT_RES=os.path.join(WORK,'results')
os.makedirs(OUT_MODELS,exist_ok=True); os.makedirs(OUT_RES,exist_ok=True)
if ON_KAGGLE:
    feat_cfg['features']['cqcc']['use_cached']=False  # avoid Drive dep; dim=278
    model_cfg['mlflow']['tracking_uri']=os.path.join(WORK,'mlruns')
print('use_cached:',feat_cfg['features']['cqcc']['use_cached'])


In [ ]:
import glob, librosa, numpy as np
from src.features.mfcc import extract_mfcc
from src.features.lfcc import extract_lfcc
from src.features.spectral import extract_spectral
from src.features.fusion import fuse_features_for_file
from src.utils.cache_loader import load_cached_cqcc
EXT={'mfcc':extract_mfcc,'lfcc':extract_lfcc,'spectral':extract_spectral}
bona=sorted(glob.glob(os.path.join(ROOT,'datasets/smoke/bonafide/*.flac')))[:50]
spoof=sorted(glob.glob(os.path.join(ROOT,'datasets/smoke/spoof/*.flac')))[:50]
files=bona+spoof; labels=[0]*len(bona)+[1]*len(spoof]
cache={}
if feat_cfg['features']['cqcc']['use_cached']:
    cache=load_cached_cqcc(os.path.join(ROOT,'datasets/smoke/cqcc_cache_smoke.h5'),[os.path.basename(p) for p in files])
print(f'files={len(files)} cache_hits={len(cache)}')
X,y=[],[ ]
import os as _os
for p,l in zip(files,labels):
    a,sr=librosa.load(p,sr=feat_cfg['sample_rate'])
    v=fuse_features_for_file(a,sr,feat_cfg,EXT,cqcc_mat=cache.get(_os.path.basename(p)))
    X.append(v); y.append(l)
import numpy as _np; X=_np.array(X); y=_np.array(y)
print('X',X.shape,'NaNs',int(_np.isnan(X).sum()))


In [ ]:
from src.models.train_xgb import tune_and_train
import json
best,metrics,path=tune_and_train(X,y,feat_cfg,model_cfg,n_trials=5,output_dir=OUT_MODELS,experiment='smoke')
print(metrics); print('saved',path)
open(os.path.join(OUT_RES,'smoke_metrics.json'),'w').write(json.dumps(metrics,indent=2))


In [ ]:
import pickle, numpy as np
m=pickle.load(open(os.path.join(OUT_MODELS,'xgb_smoke.pkl'),'rb'))['pipeline']
print('reload dim ok:',m.n_features_in_==X.shape[1])
try:
    import shap; e=shap.TreeExplainer(m.named_steps['clf']); s=e.shap_values(X[:10]); print('SHAP ok',np.shape(s))
except Exception as ex: print('SHAP skipped:',ex)
